# Data setup notebook

In [1]:
from pathlib import Path
import os
import subprocess
import sys

def local_run():
    if os.path.exists("/content"):
        # running in Google Colab
        return False
    else:
        # running locally
        return True

def get_base_dir():
    if local_run():
        # running locally
        return Path(__vsc_ipynb_file__).resolve().parent.parent
    else:
        # running in Google Colab
        return Path("/content/deforestation-unet")

if not local_run() :
    import getpass
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
if not local_run():
    token = getpass.getpass("Enter your GitHub token: ")
    base_dir = get_base_dir()
    repo_url = f"https://{token}@github.com/barbara-barta/deforestation-unet.git"
    if not base_dir.exists():
        print("Repository does not exist, cloning it now.")
        subprocess.run(["git", "clone", repo_url], check=True)
    else:
        print("Repository already exists, pulling the latest changes.")
        subprocess.run(["git", "-C", str(base_dir), "pull"], check=True)

Enter your GitHub token: ··········
Repository does not exist, cloning it now.


In [4]:
if not local_run():
    %pip install -r /content/deforestation-unet/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.8/253.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.2 MB/s eta 0:00:00


If this is your first run, please restart the kernel after all the packages are downloaded.

In [10]:
paths = {
    "RGB": base_dir / "data" / "RGB" /"raw",
    "AM4": base_dir / "data" / "AM4" / "raw",
    "AT4": base_dir / "data" / "AT4" / "raw",
}

downloads = {
    "RGB": ["https://doi.org/10.5281/zenodo.3233081", "Amazon Forest Dataset.rar"],
    "AM4": ["https://doi.org/10.5281/zenodo.4498086","AMAZON.rar"],
    "AT4": ["https://doi.org/10.5281/zenodo.4498086","ATLANTIC FOREST.rar"]}

for path in paths.values():
    path.mkdir(parents=True, exist_ok=True)

for dataset, [url, filename] in downloads.items():
    data_present = list(Path(paths[dataset]).glob(filename))
    if not data_present:
        print(f"Downloading {dataset} data...")
        cmd = ["zenodo_get", "-o", str(paths[dataset]), "-g", filename] + [url]
        subprocess.run(cmd, check=True)
        print(f"Dataset {dataset} downloaded to {str(paths[dataset])}.")
    else:
        print(
            f"{dataset} data already exists at {str(paths[dataset])}, skipping download."
        )

    unziped_filename = filename.rsplit(".", 1)[0]
    unziped_data_present = list(Path(paths[dataset]).glob(unziped_filename))

    if not unziped_data_present:
        print(f"Unzipping {dataset} data...")
        cmd = ["unrar", "x", "-o+", str(paths[dataset] / filename), str(paths[dataset])]
        subprocess.run(cmd, check=True)
        print(f"Dataset {dataset} unzipped in {str(paths[dataset])}.")

Dataset RGB downloaded to /content/deforestation-unet/data/RGB/raw.
Unzipping RGB data...
Dataset RGB unzipped in /content/deforestation-unet/data/RGB/raw.
Dataset AM4 downloaded to /content/deforestation-unet/data/AM4/raw.
Unzipping AM4 data...
Dataset AM4 unzipped in /content/deforestation-unet/data/AM4/raw.
Dataset AT4 downloaded to /content/deforestation-unet/data/AT4/raw.
Unzipping AT4 data...
Dataset AT4 unzipped in /content/deforestation-unet/data/AT4/raw.


Here: add a code cell which renames AMAZON to Amazon, Amazon Forest Dataset to AmazonForestDataset and ATLANTIC FOREST to AtlanticForest


In [20]:
names = {"RGB" : "AmazonForestDataset",
         "AM4" : "Amazon",
         "AT4" : "AtlanticForest"
}

for dataset, filename in names.items():
  copy_path = f"/content/deforestation-unet/data/{dataset}/raw/{filename}"
  dest_path = f"/content/drive/MyDrive/deforestation-unet/{dataset}/raw/{filename}"
  !cp -r {copy_path} {dest_path}